# **UNIVERSIDADE FEDERAL DO CEARA**
---
Disciplina: Introducao a analise em Big Data

---

Professor: Luiz Alexandre

---

Alunos:
1.   Julio Cesar Gama Feitosa Freitas - 583956
2.   Vitoria Freire Rocha Teixeira de Oliveira - 587661

---
Data: 13/09/2026

# 🧪 Lab 12 — ML Preview: um modelo simples de fraude

## 🎯 Objetivo

Treinar um classificador simples (Regressão Logística) que prevê `is_fraud`, e olhar para a importância das variáveis — fechando o pipeline do curso.


In [15]:
# Instala o PySpark ANTES de montar a pipeline
!pip install pyspark --quiet

In [16]:
# Importa as bibliotecas e cria as pastas utilizadas pelo laboratorio
import os
import shutil
import duckdb

os.makedirs("bigdata/raw/customers", exist_ok=True)
os.makedirs("bigdata/raw/transactions", exist_ok=True)
os.makedirs("bigdata/silver", exist_ok=True)

# Abre uma conexao DuckDB (usada so para reconstruir a Silver)
con = duckdb.connect()

print("Ambiente preparado.")

Ambiente preparado.


## Passo 0 - Reconstruir a Silver (mesmas regras do Lab 6)

In [17]:
# Faz o upload dos CSVs brutos: customers_synthetic.csv e transactions_synthetic.csv
# from google.colab import files

uploaded = [
    "../customers_synthetic.csv",
    "../transactions_synthetic.csv",
    "../fraud_labels.csv"
]

missing = [f for f in uploaded if not os.path.exists(f)] # Verifica se todos os arquivos necessários foram carregados
if missing:
    raise FileNotFoundError("Arquivos ausentes: " + ", ".join(missing))

print("\n✓ Os 3 datasets foram encontrados.")


✓ Os 3 datasets foram encontrados.


In [18]:
# Copia os CSVs enviados para a estrutura Raw do projeto
for name in uploaded:
    if "customers_synthetic" in name:
        shutil.copy(name, "bigdata/raw/customers/customers_synthetic.csv")
    elif "transactions_synthetic" in name:
        shutil.copy(name, "bigdata/raw/transactions/transactions_synthetic.csv")

print("Arquivos Raw preparados.")

Arquivos Raw preparados.


In [19]:
# Recria a Bronze de clientes e de transacoes com as mesmas regras do Lab 6
customers_raw = "bigdata/raw/customers/customers_synthetic.csv"
transactions_raw = "bigdata/raw/transactions/transactions_synthetic.csv"

con.sql(f"""
CREATE OR REPLACE TABLE bronze_customers AS
SELECT DISTINCT
    customer_id, name, cpf, email, segment,
    CAST(credit_score AS INT) AS credit_score,
    CAST(created_at AS DATE) AS created_at
FROM read_csv_auto('{customers_raw}')
WHERE customer_id IS NOT NULL
  AND credit_score BETWEEN 300 AND 900
""")

# O CASE converte explicitamente True/False (texto) para BOOLEAN
con.sql(f"""
CREATE OR REPLACE TABLE bronze_transactions AS
SELECT DISTINCT
    transaction_id, customer_id,
    CAST(amount AS FLOAT) AS amount,
    transaction_type, status,
    CAST(risk_score AS FLOAT) AS risk_score,
    CASE WHEN is_fraud = 'True' THEN true ELSE false END AS is_fraud,
    CAST(timestamp AS TIMESTAMP) AS ts
FROM read_csv_auto('{transactions_raw}')
WHERE amount > 0
  AND customer_id IS NOT NULL
""")

con.sql("SELECT COUNT(*) AS total FROM bronze_customers").show()
con.sql("SELECT COUNT(*) AS total FROM bronze_transactions").show()

┌───────┐
│ total │
│ int64 │
├───────┤
│  9993 │
└───────┘

┌────────┐
│ total  │
│ int64  │
├────────┤
│ 100000 │
└────────┘



In [20]:
# Recria a Silver juntando transacoes com clientes e derivando as colunas de analise
con.sql("""
CREATE OR REPLACE TABLE silver_transactions AS
SELECT
  t.transaction_id, t.customer_id, t.amount, t.transaction_type,
  t.status, t.risk_score, t.is_fraud, t.ts,
  c.segment, c.credit_score,
  year(t.ts)  AS year,
  month(t.ts) AS month,
  day(t.ts)   AS day,
  dayofweek(t.ts) AS day_of_week,
  CASE
    WHEN t.amount < 100  THEN 'baixo'
    WHEN t.amount < 1000 THEN 'medio'
    ELSE 'alto'
  END AS amount_band
FROM bronze_transactions t
JOIN bronze_customers c ON t.customer_id = c.customer_id
""")

con.sql("SELECT COUNT(*) FROM silver_transactions").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       100000 │
└──────────────┘



In [21]:
# Grava a Silver em Parquet apenas para o Spark ler a seguir nesta sessao
con.sql("""
COPY silver_transactions TO 'bigdata/silver/transactions_enriched.parquet' (FORMAT PARQUET)
""")

print("Parquet temporario pronto para o Spark ler.")

Parquet temporario pronto para o Spark ler.


## Setup do Spark

A partir daqui, os passos sao os mesmos das duas rotas do lab original - so muda de onde o Parquet veio.

In [22]:
# Cria a SparkSession local e le o Parquet da Silver
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("lab12").master("local[*]").getOrCreate()

df = spark.read.parquet("bigdata/silver/transactions_enriched.parquet")

## Passo 1 - Preparar as features

In [23]:
# Importa os componentes de ML usados no pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

# Transforma segment (texto) em um indice numerico
indexer = StringIndexer(inputCol="segment", outputCol="segment_idx")

# Junta as features numericas num unico vetor, que e o formato que o MLlib espera
assembler = VectorAssembler(
    inputCols=["amount", "risk_score", "credit_score", "segment_idx"],
    outputCol="features"
)

## Passo 2 - Separar treino e teste

In [24]:
# Cria a coluna label (0/1) a partir de is_fraud - o MLlib exige esse nome/tipo
df_ml = df.withColumn("label", df.is_fraud.cast("integer"))

# Divide 80% para treino e 20% para teste, com seed fixa para reprodutibilidade
train, test = df_ml.randomSplit([0.8, 0.2], seed=42)

print("treino:", train.count(), "· teste:", test.count())

treino: 79901 · teste: 20099


## Passo 3 - Montar o pipeline e treinar

In [25]:
# Encadeia indexer -> assembler -> regressao logistica num unico Pipeline
lr = LogisticRegression(featuresCol="features", labelCol="label")
pipeline = Pipeline(stages=[indexer, assembler, lr])

modelo = pipeline.fit(train)
print("Modelo treinado.")

Modelo treinado.


## Passo 4 - Avaliar no conjunto de teste

In [26]:
# Calcula a AUC (area sob a curva ROC) nas predicoes do conjunto de teste
from pyspark.ml.evaluation import BinaryClassificationEvaluator

predicoes = modelo.transform(test)
avaliador = BinaryClassificationEvaluator(labelCol="label")
auc = avaliador.evaluate(predicoes)
print(f"AUC no teste: {auc:.3f}")

AUC no teste: 0.768


**Esperado:** um valor entre 0.5 (aleatorio) e 1.0 (perfeito) - em dados sinteticos simples, e comum ver valores altos; **nao trate isso como referencia de modelo em producao**, e so para praticar o fluxo.

*(confira apos rodar a celula acima)*

## Passo 5 - Olhar exemplos de predicao

In [27]:
# Mostra transacoes individuais com a predicao do modelo e a probabilidade associada
predicoes.select("transaction_id", "amount", "segment", "label", "prediction", "probability").show(10, truncate=False)

+--------------+---------+---------+-----+----------+------------------------------------------+
|transaction_id|amount   |segment  |label|prediction|probability                               |
+--------------+---------+---------+-----+----------+------------------------------------------+
|3             |93.935265|High-Risk|0    |0.0       |[0.9495784233506188,0.05042157664938118]  |
|7             |140.57378|Premium  |0    |0.0       |[0.9944785942804952,0.005521405719504768] |
|9             |10.0     |High-Risk|0    |0.0       |[0.9338760980125079,0.06612390198749207]  |
|14            |81.39329 |Standard |0    |0.0       |[0.975478357338241,0.024521642661759047]  |
|20            |34.814598|Premium  |0    |0.0       |[0.9950050063834911,0.004994993616508903] |
|24            |26.91095 |Premium  |0    |0.0       |[0.988864583273263,0.011135416726737013]  |
|30            |130.24501|Standard |0    |0.0       |[0.9867809580635087,0.01321904193649126]  |
|36            |89.36897 |Stan

**Observe:** a coluna `probability` mostra a confianca do modelo - e o numero que viraria "score de risco" numa API real (slide 16 do DIA 3).

*(confira apos rodar a celula acima)*

## Passo 6 - Feature importance (via coeficientes da regressao)

In [28]:
# Extrai o modelo treinado (ultimo estagio do pipeline) e seus coeficientes
lr_model = modelo.stages[-1]
coefs = lr_model.coefficients

nomes = ["amount", "risk_score", "credit_score", "segment_idx"]
for nome, coef in zip(nomes, coefs):
    print(f"{nome}: {coef:.4f}")

amount: -0.0002
risk_score: 0.0167
credit_score: 0.0001
segment_idx: 1.1849


**Compare** com as barras do slide 15 do DIA 3 - os sinais (positivo/negativo) e magnitudes devem contar uma historia parecida: variaveis de risco pesando mais.

*(confira apos rodar a celula acima)*

## Checkpoint

- [ ] Pipeline treinou sem erro
- [ ] Voce tem um valor de AUC (mesmo que nao seja perfeito)
- [ ] Voce viu exemplos reais de predicao com probabilidade
- [ ] Voce comparou os coeficientes com a intuicao do slide de feature importance

---

## Fechando o pipeline do curso

```
CSV bruto -> pasta local -> Bronze (limpo) -> Silver (enriquecido)
   -> Gold (agregado) -> Dashboard (visual) -> Modelo (preditivo)
```
---

**Fim dos 12 labs.** Volte ao `00_INDICE_LABS.md` para o resumo do pipeline completo.